# Orquestador — PM a nivel mercado

PM tiene un flujo distinto al de las demás verticales (que corre `notebooks/orquestador.ipynb`): agrega los clientes por mercado antes de entrenar, y el Excel final incluye filas de referencia (mercados que ya están en la clase alta, sin pasar por el modelo).

In [ ]:
RUTA_LOCALES = "data_potential/input/vs2/locales.csv"
RUTA_EQUIFAX = "data_potential/input/vs2/equifax.csv"
RUTA_URBANICIDAD = "data_potential/input/vs2/urbanicidad.csv"
RUTA_MAESTRO_MERCADO = "data_potential/input/Maestro_de_mercado.xlsx"
RUTA_CLUSTER_MERCADO_ID = "data_potential/input/cluster_mercado_id.csv"
RUTA_SALIDA = "data_potential/output"
RUTA_METRICAS = "data_potential/output/metricas_modelos.csv"


In [ ]:
import sys, time
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent))

from src.config import get_vertical
from src.data import fetch_clientes, fetch_puntos
from src.features import agregar_features_cliente
from src.equifax import prepare_equifax_sum
from src.pm_mercado import assign_mercado_id, aggregate_mercado, map_segmento_pm, reference_rows
from src.model import fit_potential, validation_table
from src.metrics_log import log_metrics
from src.decision import apply_decision
from src.assemble import assemble_output
from src.export import compute_feature_mean, write_excel

cfg = get_vertical("pm_mercado")
m, d, e = cfg["modelo"], cfg["decision"], cfg["export"]

## 1. Datos

In [ ]:
from google.cloud import bigquery

client = bigquery.Client(project="acpe-dev-uc-ml")

df_clientes = fetch_clientes("pm_mercado", client)
df_puntos = fetch_puntos(client)
print(f"✓ {len(df_clientes):,} clientes, {len(df_puntos):,} puntos comerciales")

locales = pd.read_csv(RUTA_LOCALES, sep=";", encoding="utf-8-sig")
equifax_sum = prepare_equifax_sum(pd.read_csv(RUTA_EQUIFAX), pd.read_csv(RUTA_URBANICIDAD))
maestro = pd.read_excel(RUTA_MAESTRO_MERCADO, usecols=range(14)).dropna(axis=1, how="all")
cluster_fallback = pd.read_csv(RUTA_CLUSTER_MERCADO_ID)

## 2-3. Features de cliente (geográficas + Equifax) y mercado_id

In [ ]:
df_features, info = agregar_features_cliente(df_clientes, df_puntos, locales, equifax_sum)
print(info)

df_features["mercado_id"] = assign_mercado_id(df_features, maestro, cluster_fallback, id_col="cod_cliente_dex")
df_features = df_features.rename(columns={"cod_cliente_dex": "Cod Cliente DEX"})
sin_mercado = df_features["mercado_id"].isna().sum()
print(f"Clientes sin mercado_id (se excluyen): {sin_mercado:,} de {len(df_features):,}")
df_con_mercado = df_features.dropna(subset=["mercado_id"]).copy()

## 3b. Agregación por mercado

In [ ]:
ub_mercado = aggregate_mercado(df_con_mercado, equifax_sum, equifax_radio_km=cfg["agregacion"]["equifax_radio_km"])
ub_mercado = map_segmento_pm(ub_mercado)  # 4 clases -> 2, recién ahora (igual que el notebook original)
print(f"✓ {len(ub_mercado):,} mercados")
print(ub_mercado["segmento_trx"].value_counts())

## 4. Modelo

In [ ]:
resultado = fit_potential(
    ub_mercado, m["features"], m["categoricas"], cfg["class_order"], m["lgbm"],
    id_col="mercado_id", cluster_col=m["cluster_col"], split_valid=m["split_validacion"],
    optuna_trials=m["optuna"]["n_trials"] if m["optuna"]["activo"] else None,
)
print(f"✓ {len(resultado.estimates):,} mercados estimados")
log_metrics("pm_mercado", resultado, RUTA_METRICAS)
validation_table(resultado)

## 5. Decisión

In [ ]:
estimaciones = apply_decision(resultado.estimates, d["regla"], cfg["class_order"], d["high"], d["medium"])
print(f"Potenciales: {int(estimaciones['flag_potential'].sum()):,} de {len(estimaciones):,}")

## 6. Ensamblado + filas de referencia, Feature mean y Excel

A diferencia de las demás verticales, PM agrega los mercados que ya están en la clase alta (sin modelo) como filas de referencia — así sale el Excel original (`format_potential_df`, no `_2`).

In [ ]:
salida_estimada = assemble_output(estimaciones, ub_mercado, m["features"], id_col="mercado_id", geo_cols=("des_ciudad",))
ref = reference_rows(ub_mercado, cfg["class_order"], id_col="mercado_id", features=m["features"])
salida = pd.concat([salida_estimada, ref], ignore_index=True)
print(f"Filas estimadas: {len(salida_estimada):,} + de referencia: {len(ref):,} = {len(salida):,}")

fm = cfg["feature_mean"]
feature_mean = compute_feature_mean(salida, cfg["class_order"], fm["rmv_feats"], fm["modo"])

ids_mercado = df_con_mercado[["Cod Cliente DEX", "mercado_id"]].drop_duplicates()
diccionario = pd.read_csv(Path("..") / "configs" / "diccionario_encuesta.csv")

fecha = time.strftime("%Y%m%d")
ruta = Path(RUTA_SALIDA) / f"{fecha}_{cfg['salida']}.xlsx"
ruta.parent.mkdir(parents=True, exist_ok=True)
write_excel(
    str(ruta),
    {e["hoja_principal"]: salida, "IDs mercado": ids_mercado,
     "Diccionario Encuesta": diccionario, "Feature mean": feature_mean},
    hojas_con_formato=[e["hoja_principal"]] if e["con_formato"] else [],
)
print(f"✓ Guardado en {ruta}")